In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Fixed-key Flow watermark V1

Run all once. A single fixed 11-window key marker is generated afresh on four predeclared development prompt/seed cases. Two OFF cases calibrate the full search family; two cases evaluate OFF, SINGLE46 and MULTI44_46. FULL, DELETE90 and SPEED5_4 are all retained: 8 arms, 24 saved views, 96 encodes. This is a new, unrun existence candidate; historical attribution success does not validate it. GPU execution is performed by the user.

In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = '14596cf6301cd11242400ed617254ff3e537f65b'
DRIVE_ROOT = Path('/content/drive/MyDrive/Video-WM/Flow-Fixed-Key-V1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE_ROOT / ('flow_fixed_key_v1_' + stamp)
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(source_commit=SOURCE_SHA, python=sys.version, executable=sys.executable, status='SETUP_STARTED'), indent=2))
print('same-run output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys, json
SETUP_LOG = OUTPUT / 'setup.log'
def logged_run(command, check=True):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        (OUTPUT / 'setup_failure.json').write_text(json.dumps(dict(command=command, returncode=returncode), indent=2))
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)
print('Python:', sys.version, 'Executable:', sys.executable, flush=True)
import importlib.metadata, subprocess, sys
print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'], check=True)
logged_run(['apt-get', 'update', '-qq'], check=True)
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
print('torch before install:', version('torch'), flush=True)
if version('torch') != '2.11.0+cu128':
    logged_run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
logged_run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
check_code = """
import importlib.metadata, sys, torch, diffusers
print('Fresh process Python:', sys.version, flush=True)
print('Fresh process executable:', sys.executable, flush=True)
for name in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):
    try:
        value = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        value = None
    print(name + ':', value, flush=True)
assert str(torch.__version__) == '2.11.0+cu128', torch.__version__
assert diffusers.__version__ == '0.40.0', diffusers.__version__
"""
check_code = check_code.replace("assert str(torch.__version__)", "from pathlib import Path\ninfo = dict(python=sys.version, executable=sys.executable, packages={})\nfor package in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):\n    try: info['packages'][package] = importlib.metadata.version(package)\n    except importlib.metadata.PackageNotFoundError: info['packages'][package] = None\nimport json\nPath(RECEIPT_PATH).write_text(json.dumps(info, indent=2))\n".replace('RECEIPT_PATH', repr(str(OUTPUT / 'environment_setup.json'))) + "assert str(torch.__version__)")
logged_run([sys.executable, '-u', '-c', check_code], check=True)


In [ ]:
import subprocess
REPO = Path('/content/SC-SSTW-Fixed-Key-' + stamp)
logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_SHA
print('source commit:', actual, flush=True)


In [ ]:
import subprocess, sys
check_code = """
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('This fixed real Wan run requires a CUDA runtime')
print('device:', torch.cuda.get_device_name(0), flush=True)
"""
logged_run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import os, subprocess, sys
CONFIG = REPO / 'experiments/wan_state_clock/configs/flow_fixed_key_v1.json'
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.flow_fixed_key_run', '--config', str(CONFIG), '--output', str(OUTPUT)]
print('fixed experiment output:', OUTPUT, flush=True)
subprocess.run(command, cwd=REPO, env=env, check=True)


In [ ]:
import json
result = json.loads((OUTPUT / 'result.json').read_text())
print('status:', result['status'])
print('fixed denominator:', result['fixed_denominator'])
print('calibration:', result['calibration'])
for case_id, case in result['cases'].items():
    print(case_id, case['status'])
    for arm, item in case['videos'].items():
        print(arm, 'source:', item.get('decision'))
        for view, row in item['views'].items():
            alignment = row.get('alignment_reporting_only', {})
            summary = {k: alignment.get(k) for k in ('reference_eligible_windows', 'exact_allocation_matches', 'best_missing_reference_count', 'nominal_scale_match')}
            print(' ', view, row.get('decision'), 'alignment:', summary)
print('full result:', OUTPUT / 'result.json')
